In [1]:
# CASSANDRA setup cell — run this FIRST, in order, before any other cell.
# On Colab with GPU desired (Pythia extraction): Runtime > Change runtime
# type > GPU, THEN run this cell.
import importlib
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "{repo_url}"

if IN_COLAB:
    if not Path("/content/CASSANDRA/.git").exists():
        subprocess.run(["git", "clone", REPO_URL, "/content/CASSANDRA"],
                       check=True)
    os.chdir("/content/CASSANDRA")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt"], check=True)
    subprocess.run([sys.executable, "-m", "spacy", "download",
                    "en_core_web_sm"], check=True, capture_output=True)
    subprocess.run([sys.executable, "scripts/prepare_data.py",
                    "--config", "config.yaml"], check=True)
    print("Colab setup complete. Working directory:", os.getcwd())
else:
    try:
        import torch, transformer_lens, sklearn, pandas, seaborn  # noqa: F401
        print("Local environment OK. Run `make data` if data/processed "
              "is empty.")
    except ImportError as exc:
        raise SystemExit(
            f"Missing dependency ({{exc}}). Run `make setup` from the "
            "repository root before executing this notebook."
        )


/home/z/my-project/cassandra/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Local environment OK. Run `make data` if data/processed is empty.


# 01 — Data exploration

**Objective.** Characterize the Azaria & Mitchell true/false dataset before any modeling: domain balance, class balance, entity structure and statement lengths.

**Why it matters.** The evaluation splits group by entity and stratify by (domain, label). This notebook shows the real distribution behind that design: how many entities each domain has, how large they are, and which entities cross domain boundaries.

**Data.** `data/processed/` (statements with statement, label, domain, entity) and the fixed folds in `data/processed/folds.json`.

## Environment and determinism

The setup cell above adapts to local and Colab environments. Seed 42 is fixed here and reused by every random draw below, including the sampled examples.

In [2]:
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

pd.set_option("display.max_colwidth", 120)


In [3]:
import sys

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from cassandra_ns.report.figures import save_figure

if "google.colab" not in sys.modules:
    matplotlib.use("Agg")  # headless execution; Colab keeps its inline backend

sns.set_theme(context="notebook", style="whitegrid")
plt.rcParams["figure.dpi"] = 110


## Load the processed dataset and folds

In [4]:
from cassandra_ns.config import load_config
from cassandra_ns.data.loader import load_processed
from cassandra_ns.data.splits import load_folds

CFG = load_config("config.yaml")
df = load_processed(CFG)
folds = load_folds(CFG)

df.head()

,id,statement,label,domain,entity
0,animals:00000,The hedgehog uses walking for locomotion.,True,animals,hedgehog
1,animals:00001,"The kangaroo has a pouch for carrying and nursing its young, called joeys.",True,animals,kangaroo
2,animals:00002,The giant anteater has the atomic number of fish.,False,animals,giant anteater
3,animals:00003,"Human uses for slug include conservation, research, zoos.",False,animals,human uses for slug
4,animals:00004,"The rat feeds primarily on carrion, and has a bald head to prevent bacteria build-up while feeding.",False,animals,rat


## Per-domain summary

Statement counts, the fraction labelled true, distinct entities and mean statement length per domain. Domains with few entities are the fragile ones for grouped cross-validation: a single fold may hold very few subjects of that domain.

In [5]:
from cassandra_ns.data.eda import summary_table

summary = summary_table(df)
summary

,domain,n,pct_true,n_entities,mean_length_chars
0,animals,1008,0.5000,210,49.3
1,cities,1458,0.5000,479,31.6
2,companies,1200,0.5000,207,71.9
3,elements,930,0.5000,94,43.8
4,facts,613,0.4992,349,55.0
5,inventions,876,0.5297,233,40.8


## Class balance by domain

Grouped bars of true vs false statements per domain. A strongly imbalanced domain would let a majority-class probe look good, which is why the split stratifies by (domain, label).

In [6]:
from cassandra_ns.report.figures import save_figure

counts = (df.groupby(["domain", "label"]).size().unstack("label")
            .reindex(columns=[True, False], fill_value=0))

fig, ax = plt.subplots(figsize=(9, 4.2))
counts.plot(kind="bar", ax=ax, color=["#3a7d5c", "#b0553a"])
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_xlabel("domain")
ax.set_ylabel("statements")
ax.legend(title="label", labels=["true", "false"])
fig.tight_layout()
save_figure(fig, "01_class_balance_by_domain")
plt.show()

## Statement length by domain

Length distribution (characters) per domain as a box plot. Length is a stylistic confound: if false statements were systematically shorter or longer, a superficial probe could exploit it, so the probe evaluation adds an explicit bag-of-words control (notebook 03).

In [7]:
df["length"] = df["statement"].str.len()

fig, ax = plt.subplots(figsize=(9, 4.2))
sns.boxplot(data=df, x="domain", y="length", hue="label", ax=ax,
            palette={True: "#3a7d5c", False: "#b0553a"})
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_xlabel("domain")
ax.set_ylabel("statement length (characters)")
ax.legend(title="label", labels=["true", "false"])
fig.tight_layout()
save_figure(fig, "01_statement_length_by_domain")
plt.show()

/tmp/ipykernel_2818/328830428.py:6: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")


## Entity overlap between domains

Each cell counts entities that appear in both the row and the column domain. Shared entities are the reason row-wise splits are invalid here: the same subject in train and test folds would leak identity, not truth.

In [8]:
from cassandra_ns.data.eda import overlap_matrix

overlap = overlap_matrix(df)
overlap

,animals,cities,companies,elements,facts,inventions
animals,210,0,0,0,0,0
cities,0,479,0,0,0,0
companies,0,0,207,0,1,0
elements,0,0,0,94,3,0
facts,0,0,1,3,349,0
inventions,0,0,0,0,0,233


In [9]:
fig, ax = plt.subplots(figsize=(6.4, 5.2))
sns.heatmap(overlap, annot=True, fmt="d", cmap="viridis", ax=ax)
ax.set_xlabel("domain")
ax.set_ylabel("domain")
fig.tight_layout()
save_figure(fig, "01_entity_overlap")
plt.show()

## Entity size distribution

Statements per entity. The left tail (entities with a single statement) is what entity grouping protects: with row-wise splits those subjects scatter across folds and the probe can memorize identities. This distribution is the empirical justification for grouped cross-validation.

In [10]:
entity_sizes = df.groupby(["domain", "entity"]).size()

fig, ax = plt.subplots(figsize=(9, 4.2))
sns.boxplot(x=entity_sizes.index.get_level_values("domain"),
            y=entity_sizes.values, ax=ax, color="#4c78a8")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.set_xlabel("domain")
ax.set_ylabel("statements per entity")
fig.tight_layout()
save_figure(fig, "01_entity_size_distribution")
plt.show()

/tmp/ipykernel_2818/3359812185.py:6: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")


In [11]:
entity_sizes.describe().to_frame("statements per entity")

,statements per entity
count,1572.000000
mean,3.870865
std,2.775356
min,1.000000
25%,2.000000
50%,3.000000
75%,6.000000
max,21.000000


## Sampled statements per domain

Five random rows per domain with a fixed seed, for a direct look at the data the probes will read.

In [12]:
samples = (df.groupby("domain", group_keys=False)
             .apply(lambda g: g.sample(min(5, len(g)), random_state=SEED)))
samples[["domain", "label", "entity", "statement"]].reset_index(drop=True)

/tmp/ipykernel_2818/3503692213.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(min(5, len(g)), random_state=SEED)))


,domain,label,entity,statement
0,animals,True,armadillo,The armadillo is a mammal.
1,animals,False,goat,The goat has a habitat of various.
2,animals,True,human uses for tapir,"Human uses for tapir include conservation, research, zoos."
3,animals,False,tiger,The tiger has a habitat of grassland.
4,animals,False,dolphin,The dolphin has the atomic number of crustacean.
5,cities,False,brussels,Brussels is a name of a country.
6,cities,False,australia,Australia is a name of a city.
7,cities,True,naypyidaw,Naypyidaw is a name of a city.
8,cities,True,beirut,Beirut is a name of a city.
9,cities,True,trinidad and tobago,Trinidad and Tobago is a name of a country.


## Fold composition

The five folds are fixed on disk (folds.json carries its own sha256). Each fold must keep both classes of every domain; the table counts test-fold statements per (fold, domain).

In [13]:
fold_of = {}
for fold in folds["folds"]:
    for sid in fold["test_ids"]:
        fold_of[sid] = fold["fold"]

df["fold"] = df["id"].map(fold_of)
fold_table = (df.groupby(["fold", "domain"]).size().unstack("domain")
                .fillna(0).astype(int))
fold_table

domain,animals,cities,companies,elements,facts,inventions
fold,,,,,,
0,231,319,238,219,116,143
1,211,272,211,149,127,191
2,193,280,225,170,157,179
3,206,309,231,212,115,171
4,167,278,295,180,98,192


## Interpretation: risks for the modeling

- **Entity grouping is not optional.** Entities repeat within domains; without grouping, train/test folds share subjects and any AUC is inflated by identity recall.
- **Small-entity domains are fragile.** Domains whose entity count is close to the fold count can place almost no entities of one class in a test fold; the stratification over (domain, label) mitigates but the summary table quantifies the margin.
- **Stylistic confounds are visible here.** The length box plot is the first place where a true/false style difference would appear; it motivates the bag-of-words control probe.
- **Class balance per domain** bounds the achievable accuracy of a trivial probe; the per-domain bar chart sets that floor.
- **The folds are an anchor.** Every experiment in this project evaluates on the same folds.json; regenerating it with the same seed reproduces it byte for byte.